# Production Metrics
Metrics for each completed run, using the readings and stops matched to runs in `03_assumptions_and_observations`.

In [1]:
import contextlib
import io
from pathlib import Path

import pandas as pd

# Run 03 (which runs 02) to get production_runs, the matched readings and stops, and run_table.
# Its printed output is hidden here.
with contextlib.redirect_stdout(io.StringIO()):
    get_ipython().run_line_magic("run", "03_assumptions_and_observations.ipynb")

## How each metric is calculated
- `cases_produced`: the last used count in the run minus the first.
- `elapsed_minutes`: run end minus run start.
- `downtime_minutes`: minutes of the run's stops inside the run, with shared time counted once.
- `operating_minutes`: elapsed minutes minus downtime minutes.
- `operating_throughput_cph`: cases produced per operating hour.
- `target_attainment_percent`: throughput as a percentage of the target rate.
- `data_quality_notes`: anything that affected the numbers.

`R104` has no end time, so it only has `cases_produced`, counted to its last reading. The other metrics need its end.

## Metrics
Saved to `data/processed/run_level_dataset.csv`.

In [2]:
recorded_at = {r["reading_id"]: ts(r["recorded_at"]) for r in counter_readings}

rows = []
for run, matched in zip(production_runs, run_table):
    row = {
        "run_id": run["run_id"],
        "equipment_id": run["equipment_id"],
        "product_code": run["product_code"],
        "run_start": run["start_time"],
        "run_end": run["end_time"],
        "target_rate_cases_per_hour": run["target_hourly_rate"],
    }
    used = [r for r in readings if r["run_id"] == run["run_id"] and r["used"] == "yes"]
    if run["end_time"] is None:
        # Cases are counted, but without an end there's no operating time, so no rate
        note = f"no end time, so cases are counted only to {used[-1]['reading_id']} at {used[-1]['time']} and there's no rate"
        stops = [e["event_id"] for e in events if e["run_id"] == run["run_id"]]
        if stops:
            note += f"; {', '.join(stops)} adds no downtime"
        rows.append({**row, "cases_produced": matched["cases_produced"], "data_quality_notes": note})
        continue

    # The metrics
    start, end = ts(run["start_time"]), ts(run["end_time"])
    elapsed = minutes(start, end)
    downtime = matched["downtime_minutes"]
    operating = elapsed - downtime
    throughput = matched["cases_produced"] / operating * 60

    # What affected them
    notes = []
    first_at, last_at = recorded_at[used[0]["reading_id"]], recorded_at[used[-1]["reading_id"]]
    if first_at > start:
        notes.append(f"counted from {used[0]['reading_id']} at {used[0]['time']}, {minutes(start, first_at):.0f} min after the start")
    if last_at < end:
        notes.append(f"counted to {used[-1]['reading_id']} at {used[-1]['time']}, {minutes(last_at, end):.0f} min before the end")
    if matched["counter_readings_not_used"]:
        notes.append("readings not used: " + matched["counter_readings_not_used"])
    notes += [f'{e["event_id"]} {e["note"]}' for e in events
              if e["run_id"] == run["run_id"] and e["used"] == "yes" and e["note"]]

    rows.append({
        **row,
        "cases_produced": matched["cases_produced"],
        "elapsed_minutes": elapsed,
        "downtime_minutes": downtime,
        "operating_minutes": operating,
        "operating_throughput_cph": throughput,
        "target_attainment_percent": throughput / run["target_hourly_rate"] * 100,
        "data_quality_notes": "; ".join(notes),
    })

metrics = pd.DataFrame(rows).astype({"cases_produced": "Int64"})
Path("data/processed").mkdir(parents=True, exist_ok=True)
metrics.round(2).to_csv("data/processed/run_level_dataset.csv", index=False)

metrics.style.hide(axis="index").format(
    {"elapsed_minutes": "{:.1f}", "downtime_minutes": "{:.1f}", "operating_minutes": "{:.1f}",
     "operating_throughput_cph": "{:.1f}", "target_attainment_percent": "{:.1f}"},
    na_rep="",
)

run_id,equipment_id,product_code,run_start,run_end,target_rate_cases_per_hour,cases_produced,elapsed_minutes,downtime_minutes,operating_minutes,operating_throughput_cph,target_attainment_percent,data_quality_notes
R100,PKG-01,FLOUR-A,2026-08-13T13:00:00Z,2026-08-13T14:00:00Z,600,420,60.0,6.0,54.0,466.7,77.8,"counted from C002 at 13:10, 10 min after the start; readings not used: C004 (duplicate), C005 (count isn't a number)"
R101,PKG-01,FLOUR-A,2026-08-13T14:10:00Z,2026-08-13T15:10:00Z,600,250,60.0,9.5,50.5,297.0,49.5,"counted to C009 at 14:40, 30 min before the end; readings not used: C010 (glitch: count dropped), C011 (glitch: count dropped); D002 starts before R101: only the overlap counts; D004 shares time with an earlier stop: counted once"
R102,PKG-02,FLOUR-B,2026-08-13T13:15:00Z,2026-08-13T14:15:00Z,420,280,60.0,5.0,55.0,305.5,72.7,"counted from C015 at 13:30, 15 min after the start"
R103,PKG-02,FLOUR-B,2026-08-13T14:30:00Z,2026-08-13T15:30:00Z,420,290,60.0,3.6,56.4,308.4,73.4,readings not used: C022 (glitch: count dropped)
R104,PKG-01,FLOUR-C,2026-08-13T15:20:00Z,,540,150,,,,,,"no end time, so cases are counted only to C013 at 15:40 and there's no rate; D010 adds no downtime"
